##load cleaned DataFrames

In [0]:
claims_df = spark.table("healthcare.claims").fillna("NA").dropDuplicates()
disease_df = spark.table("healthcare.disease").fillna("NA").dropDuplicates()
group_df = spark.table("healthcare.group").fillna("NA").dropDuplicates()
grpsubgrp_df = spark.table("healthcare.grpsubgrp").fillna("NA").dropDuplicates()
hospital_df = spark.table("healthcare.hospital").fillna("NA").dropDuplicates()
patient_df = spark.table("healthcare.patient_records").fillna("NA").dropDuplicates()
subgroup_df = spark.table("healthcare.subgroup").fillna("NA").dropDuplicates()
subscriber_df = spark.table("healthcare.subscriber").fillna("NA").dropDuplicates()

print("Cleaned data loaded for result generation")

Cleaned data loaded for result generation


####Use Case 1 — Disease with maximum claims

In [0]:
from pyspark.sql.functions import *

max_claim_disease_df = claims_df.groupBy("disease_name") \
    .count() \
    .orderBy(col("count").desc())

display(max_claim_disease_df)

disease_name,count
Galactosemia,3
Pet allergy,3
Head banging,3
Phenylketonuria,3
Glaucoma,3
Anthrax,3
Choking,2
Breast cancer,2
Measles,2
Flu,2


####Use Case 2 — Subscribers age less than 30 with subgroup

In [0]:
subscriber_under_30_df = subscriber_df.withColumn(
    "age",
    floor(months_between(current_date(), to_date(col("Birth_date"))) / 12)
).filter(
    (col("age") < 30) & (col("Subgrp_id").isNotNull())
)

display(subscriber_under_30_df)

sub _id,first_name,last_name,Street,Birth_date,Gender,Phone,Country,City,Zip Code,Subgrp_id,Elig_ind,eff_date,term_date,age
SUBID10017,Bandhu,Seth,Varughese,1996-10-15,Male,+91 0695289163,India,Chinsurah,136713,S108,N,2016-10-15,2018-06-08,29
SUBID10093,Chandavarman,Singh,Sarkar Circle,1997-05-10,Others,+91 6559031791,India,Navi Mumbai,83240,S110,N,2017-05-10,2022-08-27,29


####Use Case 3 — Group with maximum subgroups


In [0]:
group_max_subgroups_df = grpsubgrp_df.groupBy("Grp_Id") \
    .count() \
    .orderBy(col("count").desc())

display(group_max_subgroups_df)

Grp_Id,count
GRP147,2
GRP143,2
GRP104,2
GRP102,1
GRP131,1
GRP109,1
GRP127,1
GRP153,1
GRP124,1
GRP117,1


####Use Case 4 — Hospital serving most patients

In [0]:
hospital_most_patients_df = patient_df.groupBy("hospital_id") \
    .count() \
    .orderBy(col("count").desc())

display(hospital_most_patients_df)

hospital_id,count
H1017,9
H1019,8
H1001,7
H1016,6
H1009,5
H1004,4
H1003,4
H1015,4
H1012,3
H1013,3


####Use Case 5 — Subgroup subscribed most number of times

In [0]:
subgroup_most_subscribed_df = subscriber_df.groupBy("Subgrp_id") \
    .count() \
    .orderBy(col("count").desc())

display(subgroup_most_subscribed_df)

Subgrp_id,count
S104,13
S109,11
S110,11
S101,11
S103,10
S105,10
S102,10
S107,8
S106,7
S108,7


####Use Case 6 — Total number of rejected claims

In [0]:
rejected_claims_df = claims_df.filter(
    col("Claim_Or_Rejected") == "Y"
)

total_rejected_claims_df = rejected_claims_df.agg(
    count("*").alias("total_rejected_claims")
)

display(total_rejected_claims_df)

total_rejected_claims
18


####Use Case 7 — City where most claims are coming from

In [0]:
claims_city_df = claims_df.join(
    patient_df,
    claims_df.patient_id == patient_df.Patient_id,
    "inner"
).groupBy(
    patient_df.city
).count().orderBy(
    col("count").desc()
)

display(claims_city_df)

city,count
Amravati,2
Jabalpur,2
Karimnagar,2
Ghaziabad,2
Mysore,2
Bihar Sharif,2
Kamarhati,2
Morbi,2
Haridwar,1
Mira-Bhayandar,1


####Use Case 8 — Government vs private policy groups subscribed mostly

In [0]:
policy_type_df = subscriber_df.join(
    grpsubgrp_df,
    subscriber_df.Subgrp_id == grpsubgrp_df.SubGrp_ID,
    "inner"
).join(
    group_df,
    grpsubgrp_df.Grp_Id == group_df.Grp_Id,
    "inner"
).groupBy(
    group_df.Grp_Type
).count().orderBy(
    col("count").desc()
)

display(policy_type_df)

Grp_Type,count
Private,338
Govt.,35


####Use Case 9 — Average monthly premium subscriber pays

In [0]:
avg_monthly_premium_df = subscriber_df.join(
    subgroup_df,
    subscriber_df.Subgrp_id == subgroup_df.SubGrp_id,
    "inner"
).agg(
    round(avg(col("Monthly_Premium")), 2).alias("average_monthly_premium")
)

display(avg_monthly_premium_df)

average_monthly_premium
1867.35


####Use Case 10 — Which group is most profitable

In [0]:
claim_amount_fixed_df = claims_df.withColumn(
    "claim_amount_num",
    regexp_extract(col("claim_amount").cast("string"), r"\+?(\d+)-", 1).cast("double")
)

most_profitable_group_df = claim_amount_fixed_df.join(
    subscriber_df,
    claim_amount_fixed_df.SUB_ID == subscriber_df["sub _id"],
    "inner"
).join(
    grpsubgrp_df,
    subscriber_df.Subgrp_id == grpsubgrp_df.SubGrp_ID,
    "inner"
).join(
    group_df,
    grpsubgrp_df.Grp_Id == group_df.Grp_Id,
    "inner"
).groupBy(
    group_df.Grp_Id,
    group_df.Grp_Name
).agg(
    sum("claim_amount_num").alias("total_claim_amount")
).orderBy(
    col("total_claim_amount").desc()
)

display(most_profitable_group_df)

Grp_Id,Grp_Name,total_claim_amount
GRP143,Magma HDI General Insurance,2185880.0
GRP147,Raheja QBE General Insurance,1743538.0
GRP104,ICICI Prudential Life Insurance Co. Ltd.,1452657.0
GRP126,Aditya Birla Health Insurance,1227417.0
GRP123,IndiaFirst Life Insurance Co. Ltd.,958463.0
GRP133,DHFL General Insurance,958463.0
GRP103,Max Life Insurance Co. Ltd.,958463.0
GRP113,Aviva Life Insurance Company India Ltd.,958463.0
GRP108,SBI Life Insurance Co. Ltd.,924681.0
GRP138,HDFC ERGO General Insurance Company,924681.0


####Use Case 11 — Patients below age 18 admitted for cancer

In [0]:
minor_cancer_patients_df = patient_df.withColumn(
    "age",
    floor(months_between(current_date(), col("patient_birth_date")) / 12)
).filter(
    (col("age") < 18) &
    (lower(col("disease_name")).contains("cancer"))
)

display(minor_cancer_patients_df)

Patient_id,Patient_name,patient_gender,patient_birth_date,patient_phone,disease_name,city,hospital_id,age


####Use Case 12 — Cashless insurance and charges >= 50,000

In [0]:
female_over_40_df = patient_df.withColumn(
    "age",
    floor(months_between(current_date(), col("patient_birth_date")) / 12)
).filter(
    (lower(col("patient_gender")) == "female") &
    (col("age") > 40)
)

display(female_over_40_df)

Patient_id,Patient_name,patient_gender,patient_birth_date,patient_phone,disease_name,city,hospital_id,age
187158,Harbir,Female,1924-06-30,+91 0112009318,Galactosemia,Rourkela,H1001,102
112766,Brahmdev,Female,1948-12-20,+91 1727749552,Bladder cancer,Tiruvottiyur,H1016,77
133424,Ballari,Female,1969-09-25,+91 0106026841,Suicide,Bihar Sharif,H1017,56
172579,Devnath,Female,1946-05-01,+91 1868774631,Food allergy,Bidhannagar,H1019,80
130339,Aakar,Female,1925-03-05,+91 2777633911,Drug consumption,Bihar Sharif,H1000,101
114241,NA,Female,1955-01-22,+91 4146391938,Breast cancer,Ghaziabad,H1015,71
167340,NA,Female,1981-01-25,+91 2960004518,Galactosemia,Surendranagar Dudhrej,H1003,45
135184,Bhagvan,Female,1966-07-24,+91 0297693485,Dengue,Bhimavaram,H1018,59
179662,Amritkala,Female,1933-11-20,+91 0537157280,Smallpox,Meerut,H1018,92
156988,Bhagavaana,Female,1935-09-16,+91 6071745855,Breast cancer,Shahjahanpur,H1012,90


####Use Case 13 — Female patients over 40 with knee surgery in past year

In [0]:
female_knee_surgery_df = patient_df.withColumn(
    "age",
    floor(months_between(current_date(), to_date(col("patient_birth_date"))) / 12)
).filter(
    (lower(col("patient_gender")) == "female") &
    (col("age") > 40) &
    (lower(col("treatment")).contains("knee")) &
    (to_date(col("admission_date")) >= add_months(current_date(), -12))
)

display(female_knee_surgery_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8725997616743724>, line 11
      1 female_knee_surgery_df = patient_df.withColumn(
      2     "age",
      3     floor(months_between(current_date(), to_date(col("patient_birth_date"))) / 12)
   (...)
      8     (to_date(col("admission_date")) >= add_months(current_date(), -12))
      9 )
---> 11 display(female_knee_surgery_df)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92  